In [ ]:
# SEADISTUSED

# --- MUDELI SEADISTUSED ---
MODEL_NAME = "google/medgemma-27b-text-it"
HF_TOKEN   = "..." # Hugging Face'i autentimismärgis

# --- ANDMED ---
DATA_PATH = "/andmed/naidis.csv" # fail patsiendiandmetega

# --- JUHENDI FAILID ---
GUIDELINE_PDF_PATH_ET  = "/juhendid/TT-ET_juhend.pdf"
GUIDELINE_PDF_PATH_EN = "/juhendid/TT-EN_juhend.pdf"

GUIDELINE_JSON_PATH_ET = "/juhendid/JS-ET_juhend.json"
GUIDELINE_JSON_PATH_EN = "/juhendid/JS-EN_juhend.json"

GUIDELINE_TEXT_PATH_ET = "/juhendid/TE-ET_juhend.txt"
GUIDELINE_TEXT_PATH_EN = "/juhendid/TE-EN_juhend.txt"

# --- LOGIMINE ---
LOG_DIR = "/logs"

COMBINED_LOG_ET = "kombineeritud_eesti.xlsx"
COMBINED_LOG_EN = "combined_english.xlsx"

# --- MUDELI PARAMEETRID ---
TEMPERATURE = 0.0
MAX_TOKENS  = 12000
MAX_MODEL_LEN = 128000
GPU_MEMORY_UTILIZATION = 0.9
TENSOR_PARALLEL_SIZE   = 1

In [ ]:
# EESTI KEELE PROMPT(ID)

ET_SYSTEM_PROMPTS = {
    "et_prompt_1": {
        "role": "system",
        "content": """..."""
    }
}

# INGLISE KEELE PROMPT(ID) 

EN_SYSTEM_PROMPTS = {
    "en_prompt_1": {
        "role": "system",
        "content": """..."""
    }
}

In [ ]:
# IMPORDITAVAD TEEGID

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import pypdf
from pathlib import Path
import json
import torch
import pandas as pd
import openpyxl
import re
import time
import os
from datetime import datetime

In [ ]:
# OMOP CONCEPT ID-D

CYTOLOGY_PROCEDURE_CONCEPTS = {
    43531329, 3025156, 4023405, 3049361, 4062484, 3039150,
}

HPV_TEST_CONCEPTS = {
    3032431, 40764133, 40764134, 42870370,
}

HPV_POSITIVE_VALUES  = {45884084, 45877985, 45879438}
HPV_NEGATIVE_VALUES  = {45878583, 45880296, 45878588}
HPV_OBSERVATION_CONCEPTS = {4317416, 441788}

COLPOSCOPY_CONCEPTS  = {4189586}
BIOPSY_CONCEPTS      = {4175471, 4311405, 40480861, 40484883}
CONISATION_CONCEPTS  = {4181912, 4264661, 4150671}
ECC_CONCEPTS         = {4200235}
ENDOMETRIAL_BIOPSY_CONCEPTS = {4299733, 4074865}

CIN1_CONCEPTS        = {192676, 4182265}
CIN2_CONCEPTS        = {196165, 4208097}
CIN3_CONCEPTS        = {198572, 194611, 4243120, 4243874, 4331440}
CIN_UNSPECIFIED_CONCEPTS = {198572, 192367, 4093447}
CIN_ALL_CONCEPTS     = CIN1_CONCEPTS | CIN2_CONCEPTS | CIN3_CONCEPTS | CIN_UNSPECIFIED_CONCEPTS
CIS_CONCEPTS         = {194611, 4243120, 4243874}
AIS_PROXY_CONCEPTS   = {4162712, 4260390, 4304331}
UNSATISFACTORY_CONCEPTS = {4012787}
ABNORMAL_CYTOLOGY_CONCEPTS = {4171912}
HYSTERECTOMY_CONCEPTS = {4023403, 4127886, 4127887, 4294805}
SALPINGECTOMY_CONCEPTS = {4115223, 4201639, 43531512}

CYTOLOGY_RESULT_CONCEPTS_MAP = {
    4162714: "NILM",   4191603: "ASCUS",
    4013220: "LSIL",   4182265: "LSIL",
    4161591: "ASC-H",  4331440: "HSIL",
    4208097: "HSIL",   4093447: "HSIL",
    4260390: "AGC-NOS",4304331: "AGC-NOS",
    4162712: "AGC-FN", 194611:  "CIS",
    4243120: "CIS",    4243874: "CIS",
}

CIN_KEYWORDS = {
    "CIN 1":   ["cin1","cin 1","cervical intraepithelial neoplasia grade 1","mild dysplasia","mild epithelial dysplasia"],
    "CIN 2":   ["cin2","cin 2","cervical intraepithelial neoplasia grade 2","moderate dysplasia","moderate epithelial dysplasia"],
    "CIN 3":   ["cin3","cin 3","cervical intraepithelial neoplasia grade 3","severe dysplasia","cin2/3","cin 2/3"],
    "CIN 2/3": ["cin2/3","cin 2/3","cin2-3","cin 2-3"],
    "CIS":     ["carcinoma in situ","cis","ca in situ"],
    "AIS":     ["adenocarcinoma in situ","ais","adenocarcinoma in situ of cervix"],
}

CYTOLOGY_RESULT_KEYWORDS = {
    "HSIL":    ["hsil","high-grade squamous","high grade squamous","high-grade intraepithelial","carcinoma in situ"],
    "ASC-H":   ["asc-h","asch","cannot exclude hsil","cannot exclude high"],
    "AGC-FN":  ["agc-fn","agc, favor neoplastic","atypical glandular cells, favor"],
    "AGC-NOS": ["agc-nos","agc nos","atypical glandular cells of undetermined","atypical endocervical cells","atypical glandular"],
    "AIS":     ["ais","adenocarcinoma in situ"],
    "CIS":     ["cis","carcinoma in situ"],
    "LSIL":    ["lsil","low-grade squamous","low grade squamous","low-grade intraepithelial","mild dysplasia"],
    "ASCUS":   ["ascus","asc-us","atypical squamous cells of undetermined","atypical squamous cells, undetermined"],
    "NILM":    ["nilm","negative for intraepithelial","no intraepithelial","negative for malignancy","normal smear","within normal limits"],
}

CYTOLOGY_HIERARCHY = ["NILM","ASCUS","LSIL","ASC-H","AGC-NOS","AGC-FN","AIS","CIS","HSIL"]

EVENT_ORDER        = {"visit":1,"condition":2,"procedure":3,"drug":4,"measurement":5,"observation":6,"device":7}
ALLOW_DUPLICATES   = {"drug","measurement","procedure"}

In [ ]:
# FUNKTSIOONID: ANDMETE TÖÖTLUS

def extract_cytology_result(patient_df: pd.DataFrame, lang: str = "et") -> str:
    found = set()
    for _, row in patient_df.iterrows():
        cid = row.get("concept_id")
        if pd.notna(cid):
            try:
                mapped = CYTOLOGY_RESULT_CONCEPTS_MAP.get(int(cid))
                if mapped:
                    found.add(mapped)
                    continue
            except (TypeError, ValueError):
                pass

        # Kombineeritud tekst
        name = str(row.get("event_name", "")).lower()
        vsv = str(row.get("value_source_value", "")).lower()
        combined = name + " " + vsv

        for result, keywords in CYTOLOGY_RESULT_KEYWORDS.items():
            for kw in keywords:
                # Word boundary → keyword peab olema eraldi sõna
                pattern = r'\b' + re.escape(kw) + r'\b'
                if re.search(pattern, combined):
                    found.add(result)
                    break 

    if not found:
        return "teadmata" if lang == "et" else "unknown"

    # Hierarhia järgi kõrgeim tulemus tagasi
    for h in reversed(CYTOLOGY_HIERARCHY):
        if h in found:
            return h

    return "teadmata" if lang == "et" else "unknown"


def extract_hpv_status(patient_df: pd.DataFrame, lang: str = "et") -> str:
    if "concept_id" not in patient_df.columns:
        return "teadmata" if lang == "et" else "unknown"
    hpv_rows = patient_df[patient_df["concept_id"].isin(HPV_TEST_CONCEPTS)]
    if not hpv_rows.empty:
        statuses = []
        for _, row in hpv_rows.iterrows():
            val = row.get("value_as_concept_id")
            try:
                val_int = int(val)
            except (TypeError, ValueError):
                vsv = str(row.get("value_source_value","")).lower()
                if any(w in vsv for w in ["positive","detected","present"]):
                    statuses.append("pos")
                elif any(w in vsv for w in ["negative","not detected","not present"]):
                    statuses.append("neg")
                continue
            if val_int in HPV_POSITIVE_VALUES:
                statuses.append("pos")
            elif val_int in HPV_NEGATIVE_VALUES:
                statuses.append("neg")
        if statuses:
            if lang == "et":
                return "positiivne" if "pos" in statuses else "negatiivne"
            else:
                return "positive" if "pos" in statuses else "negative"
        else:
            return "dokumenteeritud, tulemus puudub" if lang == "et" else "documented, result missing"
    obs_rows = patient_df[patient_df["concept_id"].isin(HPV_OBSERVATION_CONCEPTS)]
    if not obs_rows.empty:
        return "positiivne" if lang == "et" else "positive"
    for _, row in patient_df.iterrows():
        combined = str(row.get("event_name","")).lower() + " " + str(row.get("value_source_value","")).lower()
        if "hpv" in combined or "papilloma" in combined:
            if any(w in combined for w in ["positive","detected","present"]):
                return "positiivne" if lang == "et" else "positive"
            if any(w in combined for w in ["negative","not detected","not present"]):
                return "negatiivne" if lang == "et" else "negative"
    return "teadmata" if lang == "et" else "unknown"


def extract_cin_findings(patient_df: pd.DataFrame) -> list:
    found = set()
    CIN_CONCEPT_MAP = {
        192676:"CIN 1", 196165:"CIN 2", 4182265:"CIN 1", 4208097:"CIN 2",
        194611:"CIS",   4243120:"CIS",  4243874:"CIS",
        198572:"CIN",   192367:"CIN",
    }
    if "concept_id" in patient_df.columns:
        for _, row in patient_df.iterrows():
            try:
                cid = int(row.get("concept_id",0))
            except (TypeError, ValueError):
                cid = 0
            if cid in CIN_CONCEPT_MAP:
                found.add(CIN_CONCEPT_MAP[cid])
    for _, row in patient_df.iterrows():
        combined = str(row.get("event_name","")).lower() + " " + str(row.get("value_source_value","")).lower()
        for cin_label, keywords in CIN_KEYWORDS.items():
            for kw in keywords:
                pattern = r'\b' + re.escape(kw) + r'\b'
                if re.search(pattern, combined):
                    found.add(cin_label)
                    break
    return sorted(found)


def build_structured_header(person_id, age, cohort_start_date, data_end_date, patient_df, lang="et") -> str:
    followup_days   = (data_end_date - cohort_start_date).days
    followup_months = round(followup_days / 30.44, 1)
    cytology     = extract_cytology_result(patient_df, lang)
    hpv_status   = extract_hpv_status(patient_df, lang)
    cin_findings = extract_cin_findings(patient_df)
    has_cids     = "concept_id" in patient_df.columns

    colpo_rows      = patient_df[patient_df["concept_id"].isin(COLPOSCOPY_CONCEPTS)] if has_cids else pd.DataFrame()
    colposcopy_done = not colpo_rows.empty
    colposcopy_days = None
    if colposcopy_done:
        first_colpo     = pd.to_datetime(colpo_rows["event_date"]).min()
        colposcopy_days = (first_colpo - cohort_start_date).days

    biopsy_only_concepts = BIOPSY_CONCEPTS - CONISATION_CONCEPTS
    biopsy_rows  = patient_df[patient_df["concept_id"].isin(biopsy_only_concepts)] if has_cids else pd.DataFrame()
    biopsy_done  = not biopsy_rows.empty

    cone_rows    = patient_df[patient_df["concept_id"].isin(CONISATION_CONCEPTS)] if has_cids else pd.DataFrame()
    cone_done    = not cone_rows.empty
    cone_days    = None
    if cone_done:
        first_cone = pd.to_datetime(cone_rows["event_date"]).min()
        cone_days  = (first_cone - cohort_start_date).days

    ecc_rows  = patient_df[patient_df["concept_id"].isin(ECC_CONCEPTS)] if has_cids else pd.DataFrame()
    ecc_done  = not ecc_rows.empty

    if lang == "et":
        yes_no_col = "jah"
        yes_no_no  = "ei / dokumenteerimata"
        header = (
            f"=== JUHTUM {person_id} ===\n\n"
            f"STRUKTUREERITUD KOKKUVÕTE:\n"
            f"- Patsiendi vanus sõeluuringul: {age} aastat\n"
            f"- Sõeluuringu kuupäev: {cohort_start_date.date().isoformat()}\n"
            f"- Andmete lõppkuupäev: {data_end_date.date().isoformat()}\n"
            f"- Vaatlusperiood pärast sõeluuringut: {followup_months} kuud ({followup_days} päeva)\n"
            f"- Tsütoloogia tulemus: {cytology}\n"
            f"- HPV staatus: {hpv_status}\n"
            f"- Kolposkoopia teostatud: {yes_no_col if colposcopy_done else yes_no_no}"
        )
        if colposcopy_done and colposcopy_days is not None:
            header += f" ({colposcopy_days} päeva pärast sõeluuringut)"
        header += (
            f"\n- Emakakaela biopsia teostatud: {yes_no_col if biopsy_done else yes_no_no}"
            f"\n- Konisatsioon teostatud: {yes_no_col if cone_done else yes_no_no}"
        )
        if cone_done and cone_days is not None:
            header += f" ({cone_days} päeva pärast sõeluuringut)"
        header += (
            f"\n- Abrasioon teostatud: {yes_no_col if ecc_done else yes_no_no}"
            f"\n- CIN leid: {', '.join(cin_findings) if cin_findings else 'dokumenteerimata'}"
            f"\n\nKRONOLOOGILINE SÜNDMUSTE LOEND:\n"
        )
    else:
        yes_no_col = "yes"
        yes_no_no  = "no / undocumented"
        header = (
            f"=== CASE {person_id} ===\n\n"
            f"STRUCTURED SUMMARY:\n"
            f"- Patient age at screening: {age} years\n"
            f"- Screening date: {cohort_start_date.date().isoformat()}\n"
            f"- Data end date: {data_end_date.date().isoformat()}\n"
            f"- Observation period after screening: {followup_months} months ({followup_days} days)\n"
            f"- Cytology result: {cytology}\n"
            f"- HPV status: {hpv_status}\n"
            f"- Colposcopy performed: {yes_no_col if colposcopy_done else yes_no_no}"
        )
        if colposcopy_done and colposcopy_days is not None:
            header += f" ({colposcopy_days} days after screening)"
        header += (
            f"\n- Cervical biopsy performed: {yes_no_col if biopsy_done else yes_no_no}"
            f"\n- Cone biopsy (conisation) performed: {yes_no_col if cone_done else yes_no_no}"
        )
        if cone_done and cone_days is not None:
            header += f" ({cone_days} days after screening)"
        header += (
            f"\n- Endocervical curettage (ECC) performed: {yes_no_col if ecc_done else yes_no_no}"
            f"\n- CIN finding: {', '.join(cin_findings) if cin_findings else 'undocumented'}"
            f"\n\nCHRONOLOGICAL EVENT LIST:\n"
        )
    return header


def process_day_events(day_events: pd.DataFrame, lang: str = "et"):
    cleaned = []
    seen    = set()
    for _, row in day_events.iterrows():
        key = (row["event_type"], row["event_name"])
        if row["event_type"] not in ALLOW_DUPLICATES:
            if key in seen:
                continue
            seen.add(key)
        cleaned.append(row)

    cleaned_df       = pd.DataFrame(cleaned)
    grouped_sentences = []

    for event_type, grp in cleaned_df.groupby("event_type"):
        names = sorted(set(grp["event_name"].dropna()))
        if not names:
            continue
        if lang == "et":
            if event_type == "visit":
                sentence = f"Sellel päeval toimusid visiidid: {', '.join(names)}."
            elif event_type == "condition":
                sentence = f"Tuvastati seisundid/diagnoosid: {', '.join(names)}."
            elif event_type == "procedure":
                sentence = f"Tehti protseduurid: {', '.join(names)}."
            elif event_type == "drug":
                sentence = f"Manustati ravimeid: {', '.join(names)}."
            elif event_type == "observation":
                sentence = f"Täheldati/registreeriti: {', '.join(names)}."
            elif event_type == "device":
                sentence = f"Kasutati meditsiiniseadmeid: {', '.join(names)}."
            else:
                sentence = f"{event_type}: {', '.join(names)}."
        else:
            if event_type == "visit":
                sentence = f"Visits on this day: {', '.join(names)}."
            elif event_type == "condition":
                sentence = f"Conditions/diagnoses identified: {', '.join(names)}."
            elif event_type == "procedure":
                sentence = f"Procedures performed: {', '.join(names)}."
            elif event_type == "drug":
                sentence = f"Medications administered: {', '.join(names)}."
            elif event_type == "observation":
                sentence = f"Observations/registrations: {', '.join(names)}."
            elif event_type == "device":
                sentence = f"Medical devices used: {', '.join(names)}."
            else:
                sentence = f"{event_type}: {', '.join(names)}."

        if event_type == "measurement":
            measurement_parts = []
            for _, mrow in grp.iterrows():
                name    = mrow.get("event_name")
                if pd.isna(name):
                    continue
                val_name = mrow.get("value_concept_name")
                val_num  = mrow.get("value_as_number")
                val_src  = mrow.get("value_source_value")
                val_str  = ""
                if pd.notna(val_name) and str(val_name).strip():
                    val_str = str(val_name).strip()
                elif pd.notna(val_num):
                    val_str = str(val_num)
                elif pd.notna(val_src) and str(val_src).strip():
                    val_str = str(val_src).strip()
                measurement_parts.append(f"{name}: {val_str}" if val_str else str(name))
            seen_m, unique_parts = set(), []
            for p in measurement_parts:
                if p not in seen_m:
                    seen_m.add(p)
                    unique_parts.append(p)
            prefix  = "Teostati mõõtmised: " if lang == "et" else "Measurements taken: "
            sentence = prefix + "; ".join(unique_parts) + "."

        grouped_sentences.append((event_type, sentence))

    grouped_sentences.sort(key=lambda x: EVENT_ORDER.get(x[0], 99))
    return [s for _, s in grouped_sentences]


def build_patient_case_text(person_id, patient_df, lang="et") -> str:
    first_row         = patient_df.iloc[0]
    birth_dt          = first_row["birth_datetime"]
    birth_year        = birth_dt.year
    gender            = first_row["gender"]
    cohort_start_date = patient_df["cohort_start_date"].min()
    data_end_date     = patient_df["event_date"].max()

    age_at_screening = (
        cohort_start_date.year - birth_year
        - ((cohort_start_date.month, cohort_start_date.day) < (birth_dt.month, birth_dt.day))
    )

    if lang == "et":
        gender_text = "naine" if str(gender).lower().startswith("f") else str(gender)
        intro = (
            f"{birth_year} aastal sündinud {gender_text}, "
            f"sõeluuringu hetkel {age_at_screening}-aastane.\n\n"
        )
    else:
        gender_text = "female" if str(gender).lower().startswith("f") else str(gender)
        intro = (
            f"{gender_text} born in {birth_year}, "
            f"{age_at_screening} years old at the time of screening.\n\n"
        )

    structured_header = build_structured_header(
        person_id, age_at_screening, cohort_start_date, data_end_date, patient_df, lang
    )

    timeline_blocks = []
    for event_date, day_events in patient_df.groupby("event_date"):
        if pd.isna(event_date):
            continue
        delta_days   = (event_date - cohort_start_date).days
        delta_months = round(delta_days / 30.44, 1)
        delta_years  = round(delta_days / 365.25, 2)

        if lang == "et":
            if delta_days < 0:
                day_header = f"\n{event_date.date().isoformat()} ({abs(delta_days)} päeva enne sõeluuringut)"
            else:
                day_header = (
                    f"\n{event_date.date().isoformat()} "
                    f"({delta_days} päeva / {delta_months} kuud / {delta_years} aastat alates sõeluuringust)"
                )
        else:
            if delta_days < 0:
                day_header = f"\n{event_date.date().isoformat()} ({abs(delta_days)} days before screening)"
            else:
                day_header = (
                    f"\n{event_date.date().isoformat()} "
                    f"({delta_days} days / {delta_months} months / {delta_years} years since screening)"
                )

        day_sentences = process_day_events(day_events, lang)
        if day_sentences:
            timeline_blocks.append(day_header + "\n" + "\n".join(f"- {s}" for s in day_sentences))

    return intro + structured_header  + "\n".join(timeline_blocks)

In [ ]:
# FUNKTSIOONID: JUHENDITE LAADIMINE

def load_guideline_pdf(pdf_path: str) -> str:
    reader     = pypdf.PdfReader(pdf_path)
    pages_text = []
    for page in reader.pages:
        try:
            page_text = page.extract_text() or ""
        except Exception:
            page_text = ""
        pages_text.append(page_text)
    full_text = "\n".join(pages_text)
    full_text = " ".join(full_text.split())
    return full_text


def load_guideline_json(json_path: str) -> str:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    sections = []
    for diagnosis, content in data.items():
        section_text = f"DIAGNOSIS: {diagnosis}\n"
        if isinstance(content, dict):
            for key, value in content.items():
                section_text += f"- {key.upper()}: {value}\n"
        else:
            section_text += f"{content}\n"
        sections.append(section_text)
    return "\n\n".join(sections)

def load_guideline_text(txt_path: str) -> str:
    """Laeb tavalise tekstifaili (UTF-8)."""
    with open(txt_path, "r", encoding="utf-8") as f:
        return f.read().strip()

def fix_encoding(text):
    if pd.isna(text):
        return text
    try:
        return text.encode("latin1").decode("utf-8")
    except Exception:
        return text

In [ ]:
# FUNKTSIOONID: LOGIMINE

def append_log_row(log_path: str, row: dict):
    df_row = pd.DataFrame([row])
    path   = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if log_path.endswith(".csv"):
        if path.exists():
            df_row.to_csv(log_path, mode="a", header=False, index=False)
        else:
            df_row.to_csv(log_path, index=False)
    elif log_path.endswith(".xlsx"):
        if path.exists():
            with pd.ExcelWriter(log_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                existing  = pd.read_excel(log_path)
                startrow  = len(existing) + 1
                df_row.to_excel(writer, index=False, header=False, startrow=startrow)
        else:
            df_row.to_excel(log_path, index=False)
    else:
        raise ValueError("Log path peab olema .csv või .xlsx")


def parse_verdict_et(response_text: str) -> str:
    for line in response_text.split("\n"):
        stripped = line.strip()
        if stripped.startswith("HINNANG:"):
            verdict = stripped.replace("HINNANG:", "").strip()
            if "MITTEVASTAV" in verdict:
                return "MITTEVASTAV"
            elif "OSALISELT" in verdict:
                return "OSALISELT VASTAV"
            elif "VASTAV" in verdict:
                return "VASTAV"
    return "TEADMATA"


def parse_verdict_en(response_text: str) -> str:
    for line in response_text.split("\n"):
        stripped = line.strip()
        if stripped.startswith("ASSESSMENT:"):
            verdict = stripped.replace("ASSESSMENT:", "").strip()
            if "NON" in verdict:
                return "NON-COMPLIANT"
            elif "COMPLIANT" in verdict:
                return "COMPLIANT"
    return "UNKNOWN"


def update_combined_log(combined_log_path: str, person_id, variation_key: str, verdict: str):
    """
    Uuendab kombineeritud logi faili: üks rida = üks patsient, veerud = variatsioonid.
    """
    path = Path(combined_log_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists():
        df = pd.read_excel(combined_log_path)
    else:
        df = pd.DataFrame(columns=["person_id"])

    if variation_key not in df.columns:
        df[variation_key] = None

    existing_row = df[df["person_id"] == person_id]
    if existing_row.empty:
        new_row            = {"person_id": person_id, variation_key: verdict}
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        df.loc[df["person_id"] == person_id, variation_key] = verdict

    df.to_excel(combined_log_path, index=False)

In [ ]:
# FUNKTSIOON: ÜKSIKU JUHTUMI HINDAMINE

def evaluate_case(
    llm,
    tokenizer,
    system_prompt: dict,
    guideline_text: str,
    case_text: str,
    case_id: int,
    cytology: str,
    hpv_status: str,
    variation_key: str,
    lang: str,
    log_path: str,
    combined_log_path: str,
    model_name: str,
    temperature: float = TEMPERATURE,
    max_tokens: int    = MAX_TOKENS,
):
    if lang == "et":
        guideline_section_label = "SÕELUURINGU JUHEND"
        patient_section_label   = "PATSIENDI TERVISEANDMED"
    else:
        guideline_section_label = "SCREENING GUIDELINES"
        patient_section_label   = "PATIENT HEALTH DATA"

    messages = [
        system_prompt,
        {
            "role": "user",
            "content": (
                f"======================\n"
                f"{guideline_section_label}\n"
                f"======================\n\n"
                f"{guideline_text}\n\n"
                f"======================\n"
                f"{patient_section_label}\n"
                f"======================\n\n"
                f"{case_text}"
            ),
        },
    ]

    prompt       = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    input_tokens = len(tokenizer.encode(prompt))

    sampling_params = SamplingParams(temperature=temperature, max_tokens=max_tokens)

    start_time = time.time()
    outputs    = llm.generate(prompts=[prompt], sampling_params=sampling_params)
    end_time   = time.time()

    generation_time_sec = end_time - start_time
    output_obj          = outputs[0].outputs[0]
    output_text         = output_obj.text.strip()
    output_tokens       = len(output_obj.token_ids)
    total_tokens        = input_tokens + output_tokens
    tokens_per_sec      = output_tokens / generation_time_sec if generation_time_sec > 0 else 0

    verdict = parse_verdict_et(output_text) if lang == "et" else parse_verdict_en(output_text)

    print(f"\n  📊 [{variation_key}] Patsient #{case_id} | tsütoloogia={cytology} | HPV={hpv_status}")
    print(output_text)
    print(f"     Hinnang: {verdict} | Input: {input_tokens} | Output: {output_tokens} | {tokens_per_sec:.1f} tok/s")

    log_row = {
        "timestamp":           datetime.now().isoformat(timespec="seconds"),
        "variation":           variation_key,
        "lang":                lang,
        "case_id":             case_id,
        "model_name":          model_name,
        "cytology":            cytology,
        "hpv_status":          hpv_status,
        "verdict":             verdict,
        "temperature":         temperature,
        "max_tokens":          max_tokens,
        "input_tokens":        input_tokens,
        "output_tokens":       output_tokens,
        "total_tokens":        total_tokens,
        "generation_time_sec": round(generation_time_sec, 3),
        "tokens_per_sec":      round(tokens_per_sec, 2),
        "prompt_chars":        len(prompt),
        "output_chars":        len(output_text),
        "prompt_text":         prompt,
        "output_text":         output_text,
    }
    append_log_row(log_path, log_row)
    update_combined_log(combined_log_path, case_id, variation_key, verdict)

    return output_text, verdict

In [ ]:
# MAIN

def main():
    print("=" * 80)
    print("EMAKAKAELAVÄHI SÕELUURINGU VASTAVUSE HINDAMINE")
    print("=" * 80)

    # --- HuggingFace login ---
    from huggingface_hub import login
    login(HF_TOKEN)

    # --- Mudeli laadimine ---
    print(f"\n🔄 Laadin mudelit: {MODEL_NAME}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    llm    = LLM(
        model                  = MODEL_NAME,
        max_model_len          = MAX_MODEL_LEN,
        device                 = device,
        tensor_parallel_size   = TENSOR_PARALLEL_SIZE,
        enable_prefix_caching  = True,
        gpu_memory_utilization = GPU_MEMORY_UTILIZATION,
        trust_remote_code      = True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    print("✅ Mudel laaditud.")

    # --- Juhendite laadimine ---
    print("\n🔄 Laadin juhendeid...")

    guideline_pdf_text_et = load_guideline_pdf(GUIDELINE_PDF_PATH_ET)
    print(f"   PDF teksti pikkus: {len(guideline_pdf_text_et)} märki")

    guideline_json_text_et = load_guideline_json(GUIDELINE_JSON_PATH_ET)
    print(f"   JSON teksti pikkus: {len(guideline_json_text_et)} märki")

    guideline_pdf_text_en = load_guideline_pdf(GUIDELINE_PDF_PATH_EN)
    print(f"   PDF teksti pikkus: {len(guideline_pdf_text_en)} märki")

    guideline_json_text_en = load_guideline_json(GUIDELINE_JSON_PATH_EN)
    print(f"   JSON teksti pikkus: {len(guideline_json_text_en)} märki")

    # Kolme juhendi variatsioonid mõlemale keelele

    et_guidelines = {
        "et_guideline_pdf":  load_guideline_pdf(GUIDELINE_PDF_PATH_ET),
        "et_guideline_text": load_guideline_text(GUIDELINE_TEXT_PATH_ET),
        "et_guideline_json": load_guideline_json(GUIDELINE_JSON_PATH_ET),
    }
    
    en_guidelines = {
        "en_guideline_pdf":  load_guideline_pdf(GUIDELINE_PDF_PATH_EN),
        "en_guideline_text": load_guideline_text(GUIDELINE_TEXT_PATH_EN),
        "en_guideline_json": load_guideline_json(GUIDELINE_JSON_PATH_EN),
    }

    # --- Andmete laadimine ja puhastamine ---
    print("\n🔄 Laadin andmeid...")
    timeline_df = pd.read_csv(DATA_PATH)
    df          = timeline_df.copy()
    print(f"   Algne kuju: {df.shape}")

    df["value_source_value"] = df["value_source_value"].apply(fix_encoding)
    df = df[df["concept_id"] != 0]
    df = df[~df["event_name"].str.contains("No matching concept", na=False)]

    timeline_df = df
    print(f"   Puhastatud kuju: {timeline_df.shape}")

    timeline_df["event_date"]        = pd.to_datetime(timeline_df["event_date"],        errors="coerce")
    timeline_df["cohort_start_date"] = pd.to_datetime(timeline_df["cohort_start_date"], errors="coerce")
    timeline_df["birth_datetime"]    = pd.to_datetime(timeline_df["birth_datetime"],     errors="coerce")
    timeline_df = timeline_df.sort_values(by=["person_id","event_date"])

    # --- Patsiendi juhtumite ehitamine ---
    print("\n🔄 Ehitan patsiendi juhtumeid...")
    patient_cases_et = []
    patient_cases_en = []

    for person_id, patient_df in timeline_df.groupby("person_id"):
        patient_df = patient_df.copy()

        cytology_et  = extract_cytology_result(patient_df, "et")
        hpv_status_et = extract_hpv_status(patient_df, "et")
        case_text_et = build_patient_case_text(person_id, patient_df, "et")
        patient_cases_et.append((person_id, case_text_et, cytology_et, hpv_status_et))

        cytology_en  = extract_cytology_result(patient_df, "en")
        hpv_status_en = extract_hpv_status(patient_df, "en")
        case_text_en = build_patient_case_text(person_id, patient_df, "en")
        patient_cases_en.append((person_id, case_text_en, cytology_en, hpv_status_en))

    print(f"✅ {len(patient_cases_et)} patsienti valmis.")

    # --- Logikaustade loomine ---
    os.makedirs(LOG_DIR, exist_ok=True)

    # EESTI KEELE variatsioonid

    print("\n" + "=" * 80)
    print("EESTI KEEL")
    print("=" * 80)

    et_prompt_list    = list(ET_SYSTEM_PROMPTS.items())    # [(key, prompt_dict), ...]
    et_guideline_list = list(et_guidelines.items())        # [(key, text), ...]

    for prompt_key, prompt_dict in et_prompt_list:
        for guideline_key, guideline_text in et_guideline_list:
            variation_key = f"{prompt_key}__{guideline_key}_only_header"
            log_path      = os.path.join(LOG_DIR, f"{variation_key}.xlsx")

            print(f"\n  ▶ Variatsioon: {variation_key}")

            for i, (person_id, case_text, cytology, hpv_status) in enumerate(patient_cases_et, start=1):
                evaluate_case(
                    llm               = llm,
                    tokenizer         = tokenizer,
                    system_prompt     = prompt_dict,
                    guideline_text    = guideline_text,
                    case_text         = case_text,
                    case_id           = person_id,
                    cytology          = cytology,
                    hpv_status        = hpv_status,
                    variation_key     = variation_key,
                    lang              = "et",
                    log_path          = log_path,
                    combined_log_path = COMBINED_LOG_ET,
                    model_name        = MODEL_NAME,
                )

    print("\n✅ Eesti keele hindamine lõpetatud.")

    # INGLISE KEELE variatsioond

    print("\n" + "=" * 80)
    print("ENGLISH")
    print("=" * 80)

    en_prompt_list    = list(EN_SYSTEM_PROMPTS.items())
    en_guideline_list = list(en_guidelines.items())

    for prompt_key, prompt_dict in en_prompt_list:
        for guideline_key, guideline_text in en_guideline_list:
            variation_key = f"{prompt_key}__{guideline_key}_only_header"
            log_path      = os.path.join(LOG_DIR, f"{variation_key}.xlsx")

            print(f"\n  ▶ Variation: {variation_key}")

            for i, (person_id, case_text, cytology, hpv_status) in enumerate(patient_cases_en, start=1):
                evaluate_case(
                    llm               = llm,
                    tokenizer         = tokenizer,
                    system_prompt     = prompt_dict,
                    guideline_text    = guideline_text,
                    case_text         = case_text,
                    case_id           = person_id,
                    cytology          = cytology,
                    hpv_status        = hpv_status,
                    variation_key     = variation_key,
                    lang              = "en",
                    log_path          = log_path,
                    combined_log_path = COMBINED_LOG_EN,
                    model_name        = MODEL_NAME,
                )

    print("\n✅ English evaluation complete.")
    print("\n" + "=" * 80)
    print("KÕIK VARIATSIOONID LÕPETATUD")
    print(f"  Eesti kombineeritud log:   {COMBINED_LOG_ET}")
    print(f"  English combined log:      {COMBINED_LOG_EN}")
    print(f"  Eraldi logid: {LOG_DIR}/")
    print("=" * 80)

In [ ]:
if __name__ == "__main__":
    main()